# Notebook 8: Prescriptive Optimizer Core

## 1. Introduction
This is the central engine of the Prescriptive Analytics system. While previous notebooks focused on preparing the model and defining constraints, this notebook implements the logic to **recommend the optimal resource configuration** to meet a specific production target.

### How the Optimizer Works:
1. **Target Input**: The user provides a `planned_qty` (the goal).
2. **Search Space Generation**: The engine creates a grid of all possible combinations of workers, hours, and machines within the feasible bounds defined in Notebook 7.
3. **Simulation**: For every feasible combination, the engine uses the Feature Builder (Notebook 6) to predict the output using the trained ML model.
4. **Multi-Objective Scoring**: Each configuration is scored based on two factors:
    - **Production Gap**: How close is the prediction to the target?
    - **Resource Cost**: How many resources were used? (Lower is better).
5. **Ranking**: The configurations are ranked, and the Top-N best options are recommended.

In [1]:
import pandas as pd
import numpy as np
import joblib
import itertools

# Load the saved model pipeline
MODEL_PATH = '../models/production_model.joblib'
model = joblib.load(MODEL_PATH)

print("Model loaded successfully.")

Model loaded successfully.


## 2. Integrating Building Blocks

To make the optimizer work, we must import the logic developed in Notebook 6 (Feature Builder) and Notebook 7 (Constraints).

In [2]:
# --- Logic from Notebook 6: Feature Builder ---
def build_prescriptive_features(raw_inputs):
    features = raw_inputs.copy()
    workers = max(raw_inputs.get('no_of_workers', 1), 1)
    features['machine_worker_ratio'] = raw_inputs.get('machine_count', 0) / workers
    features['avg_hours_per_worker'] = raw_inputs.get('working_hours', 0) / workers
    
    defaults = {
        'week_no': 26, 'week_of_month': 3, 'financial_month': 'Sep', 
        'bottleneck_flag': False, 'rate': 5266.92
    }
    for col, value in defaults.items():
        if col not in features: features[col] = value
    
    return pd.DataFrame([features])

# --- Logic from Notebook 7: Constraints ---
RESOURCE_CONSTRAINTS = {
    'no_of_workers': {'min': 8, 'max': 32, 'step': 1},
    'working_hours': {'min': 250.0, 'max': 1500.0, 'step': 8.0},
    'machine_count': {'min': 2, 'max': 7, 'step': 1}
}

def is_feasible(config):
    for resource, bounds in RESOURCE_CONSTRAINTS.items():
        val = config.get(resource)
        if val is None or val < bounds['min'] or val > bounds['max']:
            return False
    return True

print("Building blocks integrated successfully.")

Building blocks integrated successfully.


## 3. The Prescriptive Optimizer Implementation

The `PrescriptiveOptimizer` class handles the entire lifecycle: from generating the search space to ranking the best outcomes.

In [5]:
class PrescriptiveOptimizer:
    def __init__(self, model, constraints):
        self.model = model
        self.constraints = constraints
        self.w_gap = 1.0
        self.w_res = 0.1

    def generate_search_space(self):
        workers = range(self.constraints['no_of_workers']['min'],
                        self.constraints['no_of_workers']['max'] + 1,
                        self.constraints['no_of_workers']['step'])

        hours = np.arange(self.constraints['working_hours']['min'],
                          self.constraints['working_hours']['max'] + 0.1,
                          self.constraints['working_hours']['step'])

        machines = range(self.constraints['machine_count']['min'],
                         self.constraints['machine_count']['max'] + 1,
                         self.constraints['machine_count']['step'])

        return list(itertools.product(workers, hours, machines))

    def calculate_resource_cost(self, df):
        """Vectorized cost calculation for the entire dataframe."""
        norm_workers = df['no_of_workers'] / 32
        norm_hours = df['working_hours'] / 1500
        norm_machines = df['machine_count'] / 7
        return (norm_workers + norm_hours + norm_machines) / 3

    def optimize(self, target_qty, context_vars):
        # 1. Generate all combinations
        search_space = self.generate_search_space()

        # 2. Create a single large DataFrame for all scenarios (FAST)
        df_scenarios = pd.DataFrame(search_space, columns=['no_of_workers', 'working_hours', 'machine_count'])

        # 3. Add context variables to all rows
        for col, val in context_vars.items():
            df_scenarios[col] = val

        # 4. Calculate derived features for all rows at once (FAST)
        df_scenarios['machine_worker_ratio'] = df_scenarios['machine_count'] / df_scenarios['no_of_workers']
        df_scenarios['avg_hours_per_worker'] = df_scenarios['working_hours'] / df_scenarios['no_of_workers']

        # 5. Fill baseline defaults
        defaults = {'week_no': 26, 'week_of_month': 3, 'financial_month': 'Sep', 'bottleneck_flag': False, 'rate': 5266.92}
        for col, value in defaults.items():
            if col not in df_scenarios.columns:
                df_scenarios[col] = value

        # 6. ONE SINGLE PREDICTION for all 23,000 rows (LIGHTNING FAST)
        # Ensure column order matches the model's expected input
        # (You can check the order using model.feature_names_in_ if available)
        predictions = self.model.predict(df_scenarios)

        # 7. Vectorized Scoring
        df_scenarios['predicted_qty'] = predictions
        df_scenarios['gap'] = abs(target_qty - df_scenarios['predicted_qty'])
        df_scenarios['cost'] = self.calculate_resource_cost(df_scenarios)
        df_scenarios['score'] = (self.w_gap * df_scenarios['gap']) + (self.w_res * df_scenarios['cost'])

        return df_scenarios.sort_values('score').head(10)

optimizer = PrescriptiveOptimizer(model, RESOURCE_CONSTRAINTS)
print("High-performance Optimizer Engine initialized.")

High-performance Optimizer Engine initialized.


## 4. Running a Prescriptive Simulation

We will now set a production target and let the engine find the best way to achieve it.

In [6]:
# 1. Define Target and Context
target_production = 45  # The goal: Produce 45 shoes

context = {
    'item_name': 'DIYANA - L - BROWN',
    'factory_unit': 'Factory Unit 01',
    'product_family': 'Gents',
    'size_id': 10,
    'grade': 'A',
    'order_category': 'Others'
}

# 2. Run Optimization
recommendations = optimizer.optimize(target_production, context)

print(f"Top Recommendations to achieve {target_production} units:")
display(recommendations)

Top Recommendations to achieve 45 units:


,no_of_workers,working_hours,machine_count,item_name,factory_unit,product_family,size_id,grade,order_category,machine_worker_ratio,avg_hours_per_worker,week_no,week_of_month,financial_month,bottleneck_flag,rate,predicted_qty,gap,cost,score
283,8,626.0,3,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.375000,78.250000,26,3,Sep,False,5266.92,44.994048,0.005952,0.365302,0.042483
1196,9,586.0,4,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.444444,65.111111,26,3,Sep,False,5266.92,44.998451,0.001549,0.414448,0.042994
324,8,682.0,2,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.250000,85.250000,26,3,Sep,False,5266.92,45.018630,0.018630,0.330127,0.051643
2097,10,530.0,5,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.500000,53.000000,26,3,Sep,False,5266.92,44.978965,0.021035,0.460040,0.067039
11851,20,978.0,3,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.150000,48.900000,26,3,Sep,False,5266.92,44.988527,0.011473,0.568524,0.068325
6008,14,722.0,4,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.285714,51.571429,26,3,Sep,False,5266.92,45.022296,0.022296,0.496754,0.071972
22488,31,1346.0,2,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.064516,43.419355,26,3,Sep,False,5266.92,44.998505,0.001495,0.717266,0.073221
21433,30,1194.0,3,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.100000,39.800000,26,3,Sep,False,5266.92,44.998261,0.001739,0.720690,0.073808
15377,24,650.0,7,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.291667,27.083333,26,3,Sep,False,5266.92,45.001965,0.001965,0.727778,0.074743
18381,27,890.0,5,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.185185,32.962963,26,3,Sep,False,5266.92,44.996906,0.003094,0.717123,0.074806


## 5. Analysis of Results

Look at the top row of the results. It represents the mathematically optimal balance between production accuracy and resource efficiency. 

**Things to observe:**
1. **Predicted Qty**: How close did the AI get to the target?
2. **Resource Mix**: Did it increase workers or machines to reach the goal?
3. **Gap**: Is the gap significantly smaller than the baseline prediction from Notebook 6?